# Point Cloud to scc2020 Chain Complex

This notebook shows how to prepare a point cloud with scalar function values, convert it to a two-parameter function-Alpha or function-Rips chain complex, and save the result as an `scc2020` text file.

The Alpha workflow uses GUDHI through `pointcloud_to_scc2020.py`. The Rips workflow does not require GUDHI. Run this notebook with the `tda` Python environment.

## Input Data Format

The input file is a plain text point cloud. Blank lines and text after `#` are ignored.

For a 2D point cloud, use one point per row:

```text
x y f
```

For a 3D point cloud, use:

```text
x y z f
```

Here `f` is the scalar function value attached to the vertex. With the default setting `FUNCTION_MODE = "last"`, the last column is treated as the function value and all earlier columns are point coordinates.

The output grading has two coordinates:

```text
(geometric birth, function value birth)
```

For function-Alpha, the geometric birth is the Alpha radius. For function-Rips, the geometric birth is the Rips diameter. For both methods, the function-value birth of a simplex is the maximum of the function values on its vertices.

In [ ]:
print("Starting setup cell...", flush=True)

from pathlib import Path
import subprocess
import sys

print("Imports completed.", flush=True)

PROJECT_ROOT = Path.cwd()
CONVERTER = PROJECT_ROOT / "pointcloud_to_scc2020.py"

# Use the local tda environment when it exists. Otherwise, use the current notebook kernel.
TDA_PYTHON = Path.home() / ".venv" / "tda" / "bin" / "python"
PYTHON = TDA_PYTHON if TDA_PYTHON.exists() else Path(sys.executable)

DATA_DIR = PROJECT_ROOT / "data" / "pointcloud_examples"
OUTPUT_DIR = PROJECT_ROOT / "data" / "chain_complex_scc2020"

print("Creating data directories...", flush=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data directories are ready.", flush=True)

assert CONVERTER.exists(), f"Missing converter script: {CONVERTER}"

print("Project root:", PROJECT_ROOT)
print("Python:", PYTHON)
print("Converter:", CONVERTER)
print("Output directory:", OUTPUT_DIR)

## Check GUDHI

The Alpha conversion requires GUDHI. The Rips conversion does not. This cell checks whether the selected Python interpreter can import GUDHI.

In [ ]:
check = subprocess.run(
    [str(PYTHON), "-c", "import gudhi; print(gudhi.__version__)"],
    text=True,
    capture_output=True,
    timeout=20,
)

GUDHI_AVAILABLE = check.returncode == 0

if GUDHI_AVAILABLE:
    print("GUDHI version:", check.stdout.strip())
else:
    print("GUDHI is not available in this Python environment.")
    print("Alpha conversion will fail until GUDHI is installed.")
    print("Rips conversion can still be used.")

## Choose Input and Output Paths

Edit `INPUT_POINTCLOUD` to point to your point-cloud text file. Edit `FILTRATION_METHOD` to choose `"alpha"` or `"rips"`. The output filename is built automatically from the input filename and the chosen filtration method.

The default paths below create a small example input in `data/` and save the output under `data/chain_complex_scc2020/`.

In [ ]:
INPUT_POINTCLOUD = DATA_DIR / "two_circles_200pts_knn_function.txt"

# Choose "alpha" or "rips".
FILTRATION_METHOD = "alpha"

OUTPUT_SCC2020 = OUTPUT_DIR / f"{INPUT_POINTCLOUD.stem}.{FILTRATION_METHOD}.scc2020.txt"

# Shared complex settings.
MAX_DIM = 2

# Alpha settings. Used only when FILTRATION_METHOD == "alpha".
ALPHA_CUTOFF = None
USE_SQUARED_ALPHA = False

# Rips settings. Used only when FILTRATION_METHOD == "rips".
RIPS_CUTOFF = None

# Function-value settings.
# Keep "last" for rows such as x y f or x y z f.
FUNCTION_MODE = "last"
FUNCTION_COLUMN = None

if FILTRATION_METHOD not in {"alpha", "rips"}:
    raise ValueError("FILTRATION_METHOD must be 'alpha' or 'rips'.")

print("Input point cloud:", INPUT_POINTCLOUD)
print("Input exists:", INPUT_POINTCLOUD.exists())
print("Filtration method:", FILTRATION_METHOD)
print("Output scc2020:", OUTPUT_SCC2020)

## Create a Small Example Input

Run this cell if you want a minimal 2D example. If `INPUT_POINTCLOUD` already exists, the file is not overwritten.

In [ ]:
example_text = """# x y f
0.0 0.0 0.0
1.0 0.0 0.2
0.0 1.0 0.4
1.0 1.0 0.8
0.5 0.5 0.6
"""

if INPUT_POINTCLOUD.exists():
    print("Input file already exists; keeping it unchanged:", INPUT_POINTCLOUD)
else:
    INPUT_POINTCLOUD.write_text(example_text, encoding="utf-8")
    print("Wrote example input:", INPUT_POINTCLOUD)

print(INPUT_POINTCLOUD.read_text(encoding="utf-8"))

## Convert to an Alpha or Rips scc2020 Chain Complex

This cell calls the existing command-line converter:

```text
python pointcloud_to_scc2020.py alpha INPUT_POINTCLOUD OUTPUT_SCC2020
python pointcloud_to_scc2020.py rips INPUT_POINTCLOUD OUTPUT_SCC2020
```

Set `FILTRATION_METHOD = "alpha"` or `FILTRATION_METHOD = "rips"` in the configuration cell above. The saved file is a complete two-parameter `scc2020` chain complex over `GF(2)`.

In [ ]:
INPUT_POINTCLOUD_FOR_CONVERSION = INPUT_POINTCLOUD
if not INPUT_POINTCLOUD_FOR_CONVERSION.exists() and INPUT_POINTCLOUD_FOR_CONVERSION.suffix == "":
    txt_candidate = INPUT_POINTCLOUD_FOR_CONVERSION.with_suffix(".txt")
    if txt_candidate.exists():
        print("Input file without suffix was not found; using:", txt_candidate)
        INPUT_POINTCLOUD_FOR_CONVERSION = txt_candidate

if not INPUT_POINTCLOUD_FOR_CONVERSION.exists():
    raise FileNotFoundError(f"Input point-cloud file does not exist: {INPUT_POINTCLOUD_FOR_CONVERSION}")

OUTPUT_SCC2020 = OUTPUT_DIR / f"{INPUT_POINTCLOUD_FOR_CONVERSION.stem}.{FILTRATION_METHOD}.scc2020.txt"

cmd = [
    str(PYTHON),
    str(CONVERTER),
    FILTRATION_METHOD,
    str(INPUT_POINTCLOUD_FOR_CONVERSION),
    str(OUTPUT_SCC2020),
    "--max-dim",
    str(MAX_DIM),
    "--function-mode",
    FUNCTION_MODE,
]

if FILTRATION_METHOD == "alpha" and not GUDHI_AVAILABLE:
    raise RuntimeError(
        "Alpha conversion requires GUDHI. Install it with: python -m pip install gudhi"
    )

if FUNCTION_COLUMN is not None:
    cmd.extend(["--function-column", str(FUNCTION_COLUMN)])

if FILTRATION_METHOD == "alpha" and ALPHA_CUTOFF is not None:
    cmd.extend(["--alpha-cutoff", str(ALPHA_CUTOFF)])
if FILTRATION_METHOD == "alpha" and USE_SQUARED_ALPHA:
    cmd.append("--squared-radius")
if FILTRATION_METHOD == "rips" and RIPS_CUTOFF is not None:
    cmd.extend(["--cutoff", str(RIPS_CUTOFF)])

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

print("Saved scc2020 file:", OUTPUT_SCC2020)
print("Exists:", OUTPUT_SCC2020.exists())

## Inspect the Saved scc2020 File

The first lines should start with:

```text
scc2020
2
...
--field GF(2)
```

The size line lists the number of generators from the top chain degree down to degree zero.

In [ ]:
text = OUTPUT_SCC2020.read_text(encoding="utf-8")
lines = text.splitlines()

assert lines[0].strip() == "scc2020"
assert lines[1].strip() == "2"

print("Output path:", OUTPUT_SCC2020)
print("Number of lines:", len(lines))
print()
print("\n".join(lines[:80]))

## Common Variants

If the function value is not the last column, set `FUNCTION_COLUMN` to the zero-based column index. For example, for rows `x f y`, use:

```python
FUNCTION_COLUMN = 1
```

If the function value should be one of the coordinate columns, keep `FUNCTION_COLUMN = None` and set:

```python
FUNCTION_MODE = "x"  # or "y", "z", "zero"
```

To switch between Alpha and Rips, set:

```python
FILTRATION_METHOD = "alpha"
# or
FILTRATION_METHOD = "rips"
```

If downstream software expects squared Alpha values, set:

```python
USE_SQUARED_ALPHA = True
```

If the Alpha complex is too large, set a finite Alpha radius cutoff:

```python
ALPHA_CUTOFF = 1.0
```

If the Rips complex is too large, set a finite Rips diameter cutoff:

```python
RIPS_CUTOFF = 1.0
```